# 01 · Load → pre-label → annotate (CVAT) → augment → prepare dataset

Builds a YOLO detection dataset for the Go2 detector with two classes:

| id | name | note |
|---|---|---|
| 0 | `extinguisher` | same id as the old `fire_extinguisher_best.pt`, so `CLASSES_FILTER=0` keeps meaning "extinguisher" |
| 1 | `blue_box` | new |

**Pipeline**

1. **Load** every image under `datasets/raw_images/` (subfolders = recording sessions / locations), drop unreadable files and near-duplicate frames, normalise to JPEG.
2. **Pre-label**: extinguishers with the current model, blue boxes with an HSV colour heuristic. These are *suggestions* for CVAT, not ground truth.
3. **CVAT**: the notebook writes `images.zip` + `prelabels_yolo1.1.zip`; you correct the boxes in CVAT and export as **YOLO 1.1** to `datasets/cvat/cvat_export.zip`.
4. **Import + QA** the CVAT export (class remap by name, box validation, statistics, galleries).
5. **Split** train/val/test by session (or by chunks of consecutive frames), so near-identical frames never land on both sides.
6. **Augment** the *train split only* offline (motion blur, lighting, noise, small geometric changes), optionally merge extra datasets (e.g. the old Roboflow `FireExtinguisher-1`) into train, and write `datasets/go2_det/data.yaml`.

Run top to bottom. The first run stops at section 5 until the CVAT export exists. That is expected. After exporting from CVAT, re-run from section 5 (or just re-run all, everything is idempotent).

In [ ]:
import math, random, re, shutil, zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm

## 0 · Configuration

In [ ]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'datasets'

RAW_DIR = DATA / 'raw_images'          # <- put your images here; one subfolder per session/location is recommended
WORK_DIR = DATA / 'work'               # normalised images + pre-labels (generated)
CVAT_DIR = DATA / 'cvat'               # files to upload to CVAT, and where you save the CVAT export
CVAT_EXPORT_ZIP = CVAT_DIR / 'cvat_export.zip'
OUT_DIR = DATA / 'go2_det'             # final YOLO dataset (generated, deleted and rewritten on each run)

# Class list: index = class id. Keep 'extinguisher' at 0 for compatibility with the old model / run_yolo.sh.
CLASSES = ['extinguisher', 'blue_box']
# Other spellings that may appear in CVAT / Roboflow exports, mapped onto CLASSES (compared lower-case)
CLASS_ALIASES = {'fire extinguisher': 'extinguisher', 'fire_extinguisher': 'extinguisher',
                 'fire-extinguisher': 'extinguisher', 'blue box': 'blue_box', 'bluebox': 'blue_box',
                 'blue-box': 'blue_box'}

# --- Loading / cleaning ---
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
MAX_SIDE = None            # e.g. 1280 to downscale big phone photos; None keeps the original resolution
JPEG_QUALITY = 95
DEDUP_MAX_HAMMING = 10     # dHash distance (0..256) at or below which a frame counts as a duplicate of an earlier kept one; -1 disables

# --- Pre-labelling (suggestions for CVAT) ---
PRELABEL_MODEL = ROOT / 'models' / 'fire_extinguisher_best.pt'
PRELABEL_CONF = 0.25
PRELABEL_BLUE_HSV = True   # propose blue_box boxes from a colour threshold
BLUE_HSV_LO = (100, 120, 50)   # OpenCV HSV: H in 0..179
BLUE_HSV_HI = (130, 255, 255)
BLUE_MIN_AREA_FRAC = 0.002     # ignore blue blobs smaller than this fraction of the image

# --- Import / QA ---
MIN_BOX_PX = 4                 # drop boxes thinner than this (pixels) after clipping
IMAGES_NOT_IN_EXPORT = 'drop'  # images missing from the CVAT export: 'drop' or 'negative' (treat as background)

# --- Split ---
SPLIT_FRACS = {'train': 0.8, 'val': 0.1, 'test': 0.1}
SPLIT_GROUPING = 'auto'        # 'session' | 'chunk' | 'auto' (= session if >= 5 sessions, else chunk)
CHUNK_SIZE = 20                # consecutive frames per chunk when grouping by chunk
SEED = 0

# --- Offline augmentation (train split only) ---
AUG_COPIES = 2                 # augmented copies per train image; 0 disables

# --- Extra YOLO-format datasets merged into TRAIN only (classes remapped by name; unknown classes dropped) ---
EXTRA_DATASETS = []            # e.g. [DATA / 'FireExtinguisher-1']  (a Roboflow "YOLOv8" export folder with data.yaml)

CLS_ID = {c: i for i, c in enumerate(CLASSES)}
random.seed(SEED); np.random.seed(SEED)
print('ROOT =', ROOT)

### Helpers

In [ ]:
def norm_name(name):
    n = str(name).strip().lower()
    return CLASS_ALIASES.get(n, n)


def read_yolo(txt):
    """YOLO label file -> (N, 5) float32 [cls, cx, cy, w, h] (normalised). Missing file -> empty."""
    txt = Path(txt)
    if not txt.exists():
        return np.zeros((0, 5), np.float32)
    rows = [l.split()[:5] for l in txt.read_text().splitlines() if len(l.split()) >= 5]
    return np.array(rows, np.float32).reshape(-1, 5)


def write_yolo(txt, labels):
    txt = Path(txt)
    txt.parent.mkdir(parents=True, exist_ok=True)
    txt.write_text(''.join(f'{int(c)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n' for c, x, y, w, h in labels))


COLORS = [(0, 0, 255), (255, 128, 0), (0, 200, 0), (200, 0, 200)]

def draw(img, labels, names=CLASSES):
    img = img.copy()
    H, W = img.shape[:2]
    t = max(2, W // 400)
    for c, x, y, w, h in labels:
        c = int(c)
        x1, y1, x2, y2 = int((x - w / 2) * W), int((y - h / 2) * H), int((x + w / 2) * W), int((y + h / 2) * H)
        col = COLORS[c % len(COLORS)]
        cv2.rectangle(img, (x1, y1), (x2, y2), col, t)
        cv2.putText(img, names[c] if c < len(names) else str(c), (x1, max(15, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, max(0.5, W / 1600), col, t)
    return img


def show_grid(items, ncols=4, size=4.0, title=None):
    """items: list of (bgr_image, caption)."""
    if not items:
        print('(nothing to show)')
        return
    nrows = math.ceil(len(items) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size * 0.8), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (im, cap) in zip(axes.flat, items):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(cap, fontsize=8)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def dhash(img):
    """256-bit difference hash (16x16 gradient signs), returned as 32 uint8 bytes."""
    g = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (17, 16), interpolation=cv2.INTER_AREA)
    return np.packbits((g[:, 1:] > g[:, :-1]).flatten())

## 1 · Load images from the folder

Expected layout (subfolders are optional but strongly recommended: they are used to split the data without leakage):

```
datasets/raw_images/
  lab_corridor_2026-11-10/  *.jpg
  warehouse_run1/           *.png
  ...
```

In [ ]:
assert RAW_DIR.exists() and any(RAW_DIR.iterdir()), f'Put your images in {RAW_DIR} (one subfolder per session/location)'

paths = sorted(p for p in RAW_DIR.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXTS)
rows, hashes, bad = [], [], []
for p in tqdm(paths, desc='reading'):
    im = cv2.imread(str(p))
    if im is None:
        bad.append(p)
        continue
    rel = p.relative_to(RAW_DIR)
    rows.append(dict(src=str(p), rel=rel.as_posix(),
                     session=rel.parts[0] if len(rel.parts) > 1 else '_root',
                     h=im.shape[0], w=im.shape[1]))
    hashes.append(dhash(im))

raw = pd.DataFrame(rows)
print(f'{len(raw)} readable images in {raw.session.nunique()} session(s); {len(bad)} unreadable')
for p in bad[:20]:
    print('  unreadable:', p)
display(raw.groupby('session').size().rename('images').to_frame().T)
display(raw.groupby(['w', 'h']).size().rename('images').to_frame().T)

In [ ]:
sample = raw.sample(min(8, len(raw)), random_state=SEED)
show_grid([(cv2.imread(r.src), f'{r.rel}  {r.w}x{r.h}') for r in sample.itertuples()], title='Random raw images')

## 2 · Remove near-duplicates and normalise

Consecutive camera frames are nearly identical. Keeping them all inflates labeling effort and metrics without adding information. Within each session, a frame is dropped when its dHash is within `DEDUP_MAX_HAMMING` bits of an already-kept frame.

Kept images are re-encoded to JPEG (EXIF rotation applied, optional downscale) in `datasets/work/images/` with names derived from their relative path, so names stay **stable across re-runs** (the CVAT export keeps matching).

In [ ]:
kept, dropped_pairs = [], []
for session, g in raw.groupby('session', sort=False):
    kept_idx, kept_h = [], np.zeros((0, 32), np.uint8)
    for i in g.index:
        h = hashes[i]
        if DEDUP_MAX_HAMMING >= 0 and len(kept_h):
            d = np.unpackbits(kept_h ^ h, axis=1).sum(1)
            j = int(d.argmin())
            if d[j] <= DEDUP_MAX_HAMMING:
                dropped_pairs.append((i, kept_idx[j], int(d[j])))
                continue
        kept_h = np.vstack([kept_h, h])
        kept_idx.append(i)
    kept += kept_idx

print(f'kept {len(kept)} / {len(raw)} images ({len(raw) - len(kept)} near-duplicates dropped)')
items = []
for i, j, d in random.sample(dropped_pairs, min(4, len(dropped_pairs))):
    items += [(cv2.imread(raw.src[j]), f'kept: {raw.rel[j]}'), (cv2.imread(raw.src[i]), f'dropped: {raw.rel[i]} (d={d})')]
show_grid(items, ncols=4, title='Examples of dropped duplicates (pairs)')

In [ ]:
IMG_DIR = WORK_DIR / 'images'
shutil.rmtree(IMG_DIR, ignore_errors=True)
IMG_DIR.mkdir(parents=True)

def safe_name(rel):
    return re.sub(r'[^A-Za-z0-9_-]+', '_', str(Path(rel).with_suffix('')))

man = raw.loc[kept].copy()
man['name'] = man.rel.map(safe_name)
assert man.name.is_unique, 'name collision after sanitising paths, rename the offending files'
for r in tqdm(man.itertuples(), total=len(man), desc='normalising'):
    im = cv2.imread(r.src)
    if MAX_SIDE and max(im.shape[:2]) > MAX_SIDE:
        s = MAX_SIDE / max(im.shape[:2])
        im = cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(IMG_DIR / f'{r.name}.jpg'), im, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
man = man[['name', 'src', 'session']].sort_values('name').reset_index(drop=True)
man.to_csv(WORK_DIR / 'manifest.csv', index=False)
print(f'{len(man)} images written to {IMG_DIR}')

## 3 · Pre-label (suggestions for CVAT)

* **extinguisher**: the current `fire_extinguisher_best.pt` at a low confidence. Delete false positives in CVAT rather than drawing everything from scratch.
* **blue_box**: blue regions from an HSV threshold that fill most of their bounding rectangle. Crude on purpose: it will miss boxes in poor light and fire on other blue things. Tune `BLUE_HSV_LO/HI` on the gallery below if it is badly off.

In [ ]:
PRE_DIR = WORK_DIR / 'prelabels'
shutil.rmtree(PRE_DIR, ignore_errors=True)
PRE_DIR.mkdir(parents=True)

def blue_boxes(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    m = cv2.inRange(hsv, np.array(BLUE_HSV_LO), np.array(BLUE_HSV_HI))
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k, iterations=2)
    H, W = m.shape
    out = []
    for c in cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
        x, y, w, h = cv2.boundingRect(c)
        if w * h < BLUE_MIN_AREA_FRAC * W * H or cv2.contourArea(c) < 0.5 * w * h:
            continue
        out.append([CLS_ID['blue_box'], (x + w / 2) / W, (y + h / 2) / H, w / W, h / H])
    return out

pre = {n: [] for n in man.name}

if PRELABEL_MODEL.exists():
    from ultralytics import YOLO
    model = YOLO(str(PRELABEL_MODEL))
    model_map = {i: CLS_ID[norm_name(n)] for i, n in model.names.items() if norm_name(n) in CLS_ID}
    print('pre-label model classes:', model.names, '-> our ids', model_map)
    files = [str(IMG_DIR / f'{n}.jpg') for n in man.name]
    for res in tqdm(model.predict(files, conf=PRELABEL_CONF, stream=True, verbose=False), total=len(files), desc='YOLO pre-label'):
        name = Path(res.path).stem
        for (cx, cy, w, h), c in zip(res.boxes.xywhn.tolist(), res.boxes.cls.int().tolist()):
            if c in model_map:
                pre[name].append([model_map[c], cx, cy, w, h])
else:
    print(f'No pre-label model at {PRELABEL_MODEL}, skipping YOLO pre-labels')

if PRELABEL_BLUE_HSV and 'blue_box' in CLS_ID:
    for n in tqdm(man.name, desc='blue HSV pre-label'):
        pre[n] += blue_boxes(cv2.imread(str(IMG_DIR / f'{n}.jpg')))

for n, labels in pre.items():
    write_yolo(PRE_DIR / f'{n}.txt', labels)

cnt = pd.Series([CLASSES[int(l[0])] for v in pre.values() for l in v]).value_counts()
print('pre-label boxes per class:', cnt.to_dict(), f'| images with no suggestion: {sum(not v for v in pre.values())}')

In [ ]:
with_boxes = [n for n, v in pre.items() if v]
show_names = random.sample(with_boxes, min(12, len(with_boxes)))
show_grid([(draw(cv2.imread(str(IMG_DIR / f'{n}.jpg')), pre[n]), n) for n in show_names], title='Pre-label suggestions')

## 4 · Package for CVAT

In [ ]:
CVAT_DIR.mkdir(parents=True, exist_ok=True)
imgs = [IMG_DIR / f'{n}.jpg' for n in man.name]

with zipfile.ZipFile(CVAT_DIR / 'images.zip', 'w', zipfile.ZIP_STORED) as z:
    for p in imgs:
        z.write(p, p.name)

# CVAT "YOLO 1.1" annotation format
with zipfile.ZipFile(CVAT_DIR / 'prelabels_yolo1.1.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.writestr('obj.names', '\n'.join(CLASSES) + '\n')
    z.writestr('obj.data', f'classes = {len(CLASSES)}\nnames = data/obj.names\ntrain = data/train.txt\n')
    z.writestr('train.txt', ''.join(f'data/obj_train_data/{p.name}\n' for p in imgs))
    for p in imgs:
        z.write(PRE_DIR / f'{p.stem}.txt', f'obj_train_data/{p.stem}.txt')

for f in ['images.zip', 'prelabels_yolo1.1.zip']:
    print(f'{CVAT_DIR / f}  ({(CVAT_DIR / f).stat().st_size / 1e6:.1f} MB)')

### Annotate in CVAT

**One-time CVAT setup (local, Docker):**
```bash
git clone https://github.com/cvat-ai/cvat && cd cvat
docker compose up -d
docker exec -it cvat_server bash -ic 'python3 ~/manage.py createsuperuser'
# open http://localhost:8080 and log in
```

**Per labeling round:**
1. **Projects → +** → create a project with **two rectangle labels named exactly** `extinguisher` and `blue_box` (labels are matched by name).
2. In the project, **+ Create task** → *Select files* → upload `datasets/cvat/images.zip` → Submit. For more than ~1,000 images, set *Segment size* (e.g. 300) to get several jobs that can be split between annotators.
3. Task page → **Actions → Upload annotations** → format **YOLO 1.1** → `datasets/cvat/prelabels_yolo1.1.zip`.
4. Open each job and correct boxes. Useful keys: `N` new box, `F`/`D` next/previous frame, `Del` delete, `Ctrl+S` save.
5. **Actions → Export task dataset** → format **YOLO 1.1** (images not needed) → save the zip as `datasets/cvat/cvat_export.zip`. "Ultralytics YOLO Detection 1.0" exports are accepted too.

**Labeling guidelines (be consistent; they matter more than quantity):**
* Tight boxes around the **visible** extent of the object. Include the extinguisher hose and handle; do not include the wall bracket or shadow.
* Label an object when about 30% or more of it is visible, including partly occluded or cut off at the image border.
* Label **every** instance in the image, including far and small ones. A missed instance trains the model that the object is background.
* Do **not** label pictures or posters of extinguishers, reflections, or blue objects that are not the target box.
* Leave frames that contain no target empty: they are valuable hard negatives, especially ones with red or blue clutter.

## 5 · Import the CVAT export + QA

In [ ]:
assert CVAT_EXPORT_ZIP.exists(), (
    f'No CVAT export at {CVAT_EXPORT_ZIP}.\n'
    'Annotate in CVAT (section 4), export the task as "YOLO 1.1", save the zip there, then re-run from section 5.')

def load_yolo_zip(zip_path):
    """Returns ({image_stem: (N,5) array in OUR class ids}, export_class_names). Accepts YOLO 1.1 and Ultralytics YOLO exports."""
    with zipfile.ZipFile(zip_path) as z:
        files = z.namelist()
        names = None
        for f in files:
            base = f.rsplit('/', 1)[-1]
            if base == 'obj.names':
                names = [l.strip() for l in z.read(f).decode().splitlines() if l.strip()]
        if names is None:
            for f in files:
                if f.rsplit('/', 1)[-1] in ('data.yaml', 'dataset.yaml'):
                    n = yaml.safe_load(z.read(f))['names']
                    names = [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n)
        assert names, 'no obj.names / data.yaml in the export: export as "YOLO 1.1" or "Ultralytics YOLO Detection 1.0"'
        remap = {i: CLS_ID[norm_name(n)] for i, n in enumerate(names) if norm_name(n) in CLS_ID}
        unknown = [n for n in names if norm_name(n) not in CLS_ID]
        if unknown:
            print('WARNING: labels not in CLASSES are dropped:', unknown)
        labels = {}
        for f in files:
            if not f.endswith('.txt') or '/' not in f:   # skip root-level train.txt etc.
                continue
            rows = []
            for line in z.read(f).decode().splitlines():
                p = line.split()
                if len(p) >= 5 and int(float(p[0])) in remap:
                    rows.append([remap[int(float(p[0]))], *map(float, p[1:5])])
            labels[Path(f).stem] = np.array(rows, np.float32).reshape(-1, 5)
    return labels, names

def clean_labels(lab, W, H):
    """Clip to the image, drop degenerate / tiny / duplicate boxes. Returns (labels, n_dropped)."""
    if not len(lab):
        return lab, 0
    c, cx, cy, w, h = lab.T
    x1, y1 = np.clip(cx - w / 2, 0, 1), np.clip(cy - h / 2, 0, 1)
    x2, y2 = np.clip(cx + w / 2, 0, 1), np.clip(cy + h / 2, 0, 1)
    out = np.stack([c, (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1], 1)
    keep = ((x2 - x1) * W >= MIN_BOX_PX) & ((y2 - y1) * H >= MIN_BOX_PX)
    out = out[keep]
    out = np.unique(np.round(out, 5), axis=0)
    return out.astype(np.float32), len(lab) - len(out)

# Reload state from disk so this section also works after a kernel restart (run sections 0 + Helpers first)
IMG_DIR, PRE_DIR = WORK_DIR / 'images', WORK_DIR / 'prelabels'
man = pd.read_csv(WORK_DIR / 'manifest.csv')
pre = {n: read_yolo(PRE_DIR / f'{n}.txt').tolist() for n in man.name}

exp_labels, exp_names = load_yolo_zip(CVAT_EXPORT_ZIP)
print(f'export classes: {exp_names} | label files: {len(exp_labels)}')

in_export = man.name.isin(exp_labels.keys())
stale = sorted(set(exp_labels) - set(man.name))
print(f'images matched: {in_export.sum()}/{len(man)}; label files without a current image: {len(stale)}')
assert in_export.sum() > 0, 'no image names match the export: did you export the right task?'
if (~in_export).any():
    print(f'WARNING: {(~in_export).sum()} images are not in the export -> IMAGES_NOT_IN_EXPORT={IMAGES_NOT_IN_EXPORT!r}', list(man.name[~in_export][:5]))

ann_rows, labels, n_fixed = [], {}, 0
for r in man.itertuples():
    if r.name not in exp_labels and IMAGES_NOT_IN_EXPORT == 'drop':
        continue
    H, W = cv2.imread(str(IMG_DIR / f'{r.name}.jpg')).shape[:2]
    lab, nd = clean_labels(exp_labels.get(r.name, np.zeros((0, 5), np.float32)), W, H)
    n_fixed += nd
    labels[r.name] = lab
    ann_rows.append(dict(name=r.name, session=r.session, w=W, h=H,
                         **{f'n_{c}': int((lab[:, 0] == i).sum()) for i, c in enumerate(CLASSES)}))
ann = pd.DataFrame(ann_rows)
print(f'{len(ann)} annotated images; {n_fixed} boxes dropped/merged by cleaning')

In [ ]:
ncols = [f'n_{c}' for c in CLASSES]
summary = pd.DataFrame({
    'instances': [ann[c].sum() for c in ncols],
    'images containing': [(ann[c] > 0).sum() for c in ncols],
}, index=CLASSES)
display(summary)
neg = (ann[ncols].sum(1) == 0)
print(f'background (no-object) images: {neg.sum()} ({neg.mean():.0%}); healthy range is roughly 5-20%')
display(ann.groupby('session')[ncols].sum().assign(images=ann.groupby('session').size()))

# Agreement between pre-labels and final labels: how much the pre-labeller helped
def iou_xywh(a, b):
    a1, a2 = a[:, None, :2] - a[:, None, 2:] / 2, a[:, None, :2] + a[:, None, 2:] / 2
    b1, b2 = b[None, :, :2] - b[None, :, 2:] / 2, b[None, :, :2] + b[None, :, 2:] / 2
    inter = np.clip(np.minimum(a2, b2) - np.maximum(a1, b1), 0, None).prod(-1)
    return inter / (a[:, None, 2:].prod(-1) + b[None, :, 2:].prod(-1) - inter + 1e-9)

for i, c in enumerate(CLASSES):
    tp = n_pre = n_gt = 0
    for n, lab in labels.items():
        g, p = lab[lab[:, 0] == i, 1:], np.array([l[1:] for l in pre.get(n, []) if l[0] == i]).reshape(-1, 4)
        n_pre, n_gt = n_pre + len(p), n_gt + len(g)
        if len(g) and len(p):
            tp += int((iou_xywh(p, g).max(1) >= 0.5).sum())
    print(f'pre-label {c:>12}: {n_pre} suggested, {tp} kept (IoU>=0.5) -> precision {tp / max(n_pre, 1):.2f}, recall {tp / max(n_gt, 1):.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for i, c in enumerate(CLASSES):
    sz = [math.sqrt(l[3] * r.w * l[4] * r.h) for r in ann.itertuples() for l in labels[r.name] if l[0] == i]
    axes[0].hist(sz, bins=40, alpha=0.6, label=c)
    ar = [l[3] * r.w / max(l[4] * r.h, 1e-6) for r in ann.itertuples() for l in labels[r.name] if l[0] == i]
    axes[1].hist(ar, bins=40, alpha=0.6, label=c)
axes[0].set_title('box size sqrt(w*h) [px]  (<32 = small/far)'); axes[0].legend()
axes[1].set_title('aspect ratio w/h'); axes[1].legend()
centers = np.concatenate([labels[n][:, 1:3] for n in ann.name if len(labels[n])] or [np.zeros((0, 2))])
axes[2].hist2d(centers[:, 0], centers[:, 1], bins=20); axes[2].invert_yaxis(); axes[2].set_title('box centre heat-map')
plt.tight_layout(); plt.show()

In [ ]:
for i, c in enumerate(CLASSES):
    names_c = list(ann.name[ann[f'n_{c}'] > 0])
    pick = random.sample(names_c, min(8, len(names_c)))
    show_grid([(draw(cv2.imread(str(IMG_DIR / f'{n}.jpg')), labels[n]), n) for n in pick], title=f'Final labels: {c}')
negs = list(ann.name[neg])
show_grid([(cv2.imread(str(IMG_DIR / f'{n}.jpg')), n) for n in random.sample(negs, min(4, len(negs)))], title='Background images')

## 6 · Split train / val / test (grouped, leakage-free)

Images are grouped by **session** (subfolder), or by **chunks of `CHUNK_SIZE` consecutive frames** when there are too few sessions, and whole groups are assigned to a split. Many random group assignments are tried and the one closest to the target fractions, both overall and per class, is kept, with a large penalty when val or test is missing a class.

In [ ]:
grouping = SPLIT_GROUPING if SPLIT_GROUPING != 'auto' else ('session' if ann.session.nunique() >= 5 else 'chunk')
ann = ann.sort_values('name').reset_index(drop=True)
if grouping == 'session':
    ann['group'] = ann.session
else:
    ann['group'] = ann.session + '#' + (ann.groupby('session').cumcount() // CHUNK_SIZE).astype(str)

G = ann.groupby('group')[ncols].sum().assign(images=ann.groupby('group').size())
splits = list(SPLIT_FRACS)
target = np.array([SPLIT_FRACS[s] for s in splits])
totals = G[['images'] + ncols].sum().values.astype(float)

def score(assign):
    cost = 0.0
    for k, s in enumerate(splits):
        part = G[assign == k][['images'] + ncols].sum().values
        frac = part / np.maximum(totals, 1)
        cost += np.abs(frac - target[k]).sum()
        if s != 'train':
            cost += 10.0 * ((part[1:] == 0) & (totals[1:] > 0)).sum()   # a class missing from val/test
    return cost

rng = np.random.default_rng(SEED)
best, best_cost = None, np.inf
for _ in range(3000):
    order = rng.permutation(len(G))
    assign, filled = np.zeros(len(G), int), np.zeros(len(splits))
    for gi in order:
        k = int(np.argmax(target * totals[0] - filled))   # split with the largest remaining deficit
        assign[gi] = k
        filled[k] += G.images.iloc[gi]
    c = score(assign)
    if c < best_cost:
        best, best_cost = assign, c

ann['split'] = ann.group.map(dict(zip(G.index, [splits[k] for k in best])))
print(f'grouping by {grouping}: {len(G)} groups, split cost {best_cost:.3f}')
tab = ann.groupby('split')[ncols].sum().assign(images=ann.groupby('split').size()).loc[splits]
display(pd.concat([tab, (tab / tab.sum()).add_suffix(' %').round(2)], axis=1))

## 7 · Write the dataset, augment train offline, merge extra data

The offline augmentations simulate what the Go2 camera sees and Ultralytics' built-in online augmentation does not cover well:
motion blur from the walking gait, exposure changes, sensor noise, JPEG artefacts, slight roll and viewpoint changes.
**Hue shifts are kept tiny**, because colour (red vs blue) is part of what separates the two classes.
Ultralytics still applies its own online augmentation (mosaic, HSV, flips, scale) during training (notebook 02).

In [ ]:
import albumentations as A
print('albumentations', A.__version__)

AUG = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.85, 1.15), translate_percent=(-0.06, 0.06), rotate=(-6, 6), shear=(-3, 3),
             border_mode=cv2.BORDER_CONSTANT, p=0.6),
    A.OneOf([A.MotionBlur(blur_limit=(3, 11)), A.GaussianBlur(blur_limit=(3, 5)), A.Defocus(radius=(2, 4))], p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.RandomGamma(gamma_limit=(70, 140), p=0.3),
    A.HueSaturationValue(hue_shift_limit=4, sat_shift_limit=25, val_shift_limit=20, p=0.4),
    A.RandomShadow(p=0.15),
    A.OneOf([A.GaussNoise(std_range=(0.02, 0.06)), A.ISONoise()], p=0.3),
    A.ImageCompression(quality_range=(40, 90), p=0.3),
], bbox_params=A.BboxParams(format='yolo', label_fields=['cls'], min_visibility=0.35, clip=True), seed=SEED)

def augment(img, lab):
    out = AUG(image=img, bboxes=lab[:, 1:].tolist(), cls=lab[:, 0].tolist())
    b = np.array(out['bboxes'], np.float32).reshape(-1, 4)
    return out['image'], np.concatenate([np.array(out['cls'], np.float32).reshape(-1, 1), b], 1)

# Preview: 3 train images x (original + 3 augmentations)
items = []
prev = ann[(ann.split == 'train') & (ann[ncols].sum(1) > 0)].sample(min(3, (ann.split == 'train').sum()), random_state=SEED)
for n in prev.name:
    im, lab = cv2.imread(str(IMG_DIR / f'{n}.jpg')), labels[n]
    items.append((draw(im, lab), f'original {n}'))
    for k in range(3):
        ai, al = augment(im, lab)
        items.append((draw(ai, al), f'aug {k}'))
show_grid(items, ncols=4, title='Offline augmentation preview')

In [ ]:
shutil.rmtree(OUT_DIR, ignore_errors=True)
for s in splits:
    (OUT_DIR / 'images' / s).mkdir(parents=True)
    (OUT_DIR / 'labels' / s).mkdir(parents=True)

records = []
for r in tqdm(ann.itertuples(), total=len(ann), desc='writing'):
    shutil.copy2(IMG_DIR / f'{r.name}.jpg', OUT_DIR / 'images' / r.split / f'{r.name}.jpg')
    write_yolo(OUT_DIR / 'labels' / r.split / f'{r.name}.txt', labels[r.name])
    records.append(dict(file=f'{r.name}.jpg', split=r.split, source='raw', session=r.session, group=r.group))
    if r.split == 'train':
        im = cv2.imread(str(IMG_DIR / f'{r.name}.jpg'))
        for k in range(AUG_COPIES):
            ai, al = augment(im, labels[r.name])
            an = f'{r.name}__aug{k}'
            cv2.imwrite(str(OUT_DIR / 'images' / 'train' / f'{an}.jpg'), ai, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            write_yolo(OUT_DIR / 'labels' / 'train' / f'{an}.txt', al)
            records.append(dict(file=f'{an}.jpg', split='train', source='aug', session=r.session, group=r.group))

In [ ]:
# Optional: merge extra YOLO-format datasets (e.g. the old Roboflow FireExtinguisher-1) into TRAIN only.
# Their val/test images are also put into train: the Go2 val/test splits stay purely in-domain.
# Caveat: such datasets are only labelled for their own classes, so any blue box in them is an unlabelled "background".
for ds in EXTRA_DATASETS:
    ds = Path(ds)
    y = yaml.safe_load((ds / 'data.yaml').read_text())
    n = y['names']
    names = [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n)
    remap = {i: CLS_ID[norm_name(x)] for i, x in enumerate(names) if norm_name(x) in CLS_ID}
    print(f'{ds.name}: classes {names} -> {remap}')
    added = 0
    for img_p in sorted(p for p in ds.rglob('*') if p.suffix.lower() in IMG_EXTS and 'images' in p.parts):
        parts = list(img_p.parts)
        parts[len(parts) - 1 - parts[::-1].index('images')] = 'labels'
        lab = read_yolo(Path(*parts).with_suffix('.txt'))
        lab = np.array([[remap[int(l[0])], *l[1:]] for l in lab if int(l[0]) in remap], np.float32).reshape(-1, 5)
        dst = f'extra_{safe_name(ds.name)}__{safe_name(img_p.relative_to(ds))}'
        shutil.copy2(img_p, OUT_DIR / 'images' / 'train' / f'{dst}{img_p.suffix.lower()}')
        write_yolo(OUT_DIR / 'labels' / 'train' / f'{dst}.txt', lab)
        records.append(dict(file=f'{dst}{img_p.suffix.lower()}', split='train', source=f'extra:{ds.name}', session='', group=''))
        added += 1
    print(f'  added {added} images to train')

In [ ]:
data_yaml = {
    'path': str(OUT_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {i: c for i, c in enumerate(CLASSES)},
}
(OUT_DIR / 'data.yaml').write_text(yaml.safe_dump(data_yaml, sort_keys=False))
manifest = pd.DataFrame(records)
manifest.to_csv(OUT_DIR / 'manifest.csv', index=False)

# Final summary, re-read from disk (verifies what was actually written)
src_of = dict(zip(manifest.file.map(lambda f: Path(f).stem), manifest.source))
rows = []
for s in splits:
    for t in (OUT_DIR / 'labels' / s).glob('*.txt'):
        lab = read_yolo(t)
        rows.append(dict(split=s, source=src_of.get(t.stem, '?'),
                         **{c: int((lab[:, 0] == i).sum()) for i, c in enumerate(CLASSES)}, empty=int(len(lab) == 0)))
final = pd.DataFrame(rows)
display(final.groupby(['split', 'source']).agg(images=('empty', 'size'), background=('empty', 'sum'),
                                               **{c: (c, 'sum') for c in CLASSES}).loc[splits])
n_img = sum(1 for s in splits for _ in (OUT_DIR / 'images' / s).iterdir())
assert n_img == len(final), f'{n_img} images vs {len(final)} label files'
print(f'\nDataset ready: {OUT_DIR / "data.yaml"}')
print((OUT_DIR / 'data.yaml').read_text())

In [ ]:
train_files = sorted((OUT_DIR / 'images' / 'train').iterdir())
pick = random.sample(train_files, min(8, len(train_files)))
show_grid([(draw(cv2.imread(str(p)), read_yolo(OUT_DIR / 'labels' / 'train' / f'{p.stem}.txt')), p.name) for p in pick],
          title='Round-trip check: train images + labels read back from disk')

**Next:** `02_train_evaluate_export.ipynb`.
For later labeling rounds, add new images to `raw_images/` (new subfolder), re-run, and in CVAT create a new task for the new images only, or re-upload everything and keep exporting the full task. File names are stable, so earlier annotations keep matching.